# Test video — So sánh 2 model YOLO26 (nano vs small)

**Mục tiêu:** chạy thử 2 model đã train trên `video_test.mp4` bằng OpenCV, hiển thị song song trong 1 video và đo lường:

| Metric | Ý nghĩa |
|---|---|
| **FPS** | Số khung hình xử lý được mỗi giây (thực đo khi chạy video, có EMA làm mượt + trung bình toàn video) |
| **Threshold** | Ngưỡng confidence `conf` đang dùng để giữ bounding box |
| **Counting** | Số con gà phát hiện trong frame hiện tại (kèm max & tổng cả video) |

- **Data mới:** `Chicken Counting v18` (Roboflow → `datasets/chicken_counting_v18_yolo`, 5288 ảnh 640×640, split 3696/1098/494) — lớp `chicken`
- **Model:** `yolo26n` (nano) · `yolo26s` (small) — weight tại `models/chicken_detection/runs/*/weights/best.pt`
- **mAP@0.5 trong bảng dưới là của model cũ** (data chicken_counting_v18, đã backup) — ⚠️ **cập nhật ô `MAP50` sau khi train xong data mới** (xem `evaluation/validation_report.md`)
- **Video:** `video_test.mp4` — 1280×720, 30 fps, 927 frame (~31 s)
- **Đầu ra:** `output_compare.mp4` (nửa trái = nano, nửa phải = small) + `comparison.csv`

> Chạy bằng kernel **Python (chicken-ml)** (venv của dự án).

In [19]:
# ══ CẤU HÌNH — chỉnh ở đây rồi chạy lại là HUD tự cập nhật ══
from pathlib import Path
import time

import cv2
import numpy as np
import pandas as pd
import torch
from ultralytics import YOLO

CONF  = 0.3          # ★ THRESHOLD — ngưỡng confidence (0.05–0.9)
IOU   = 0.5          # ngưỡng NMS IoU
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SHOW_LIVE = False    # True → mở cửa sổ cv2.imshow, bấm "q" để thoát

HALF_W, HALF_H = 640, 720   # kích thước mỗi nửa khung hình (output 1280×720)


def find_ml_root(start: Path) -> Path:
    """Tìm thư mục 04_machine_learning kể cả khi cwd không phải thư mục testing."""
    for p in [start, *start.parents]:
        if p.name == "04_machine_learning":
            return p
    raise FileNotFoundError("Không tìm thấy 04_machine_learning từ " + str(start))


ML_ROOT      = find_ml_root(Path.cwd())
TESTING_DIR  = ML_ROOT / "testing"
VIDEO        = TESTING_DIR / "video_test.mp4"
OUTPUT       = TESTING_DIR / "output_compare.mp4"
CSV_OUT      = TESTING_DIR / "comparison.csv"

MODELS = {
    "yolo26n (nano)":  ML_ROOT / "models/chicken_detection/runs/yolo26n/weights/best.pt",
    "yolo26s (small)": ML_ROOT / "models/chicken_detection/runs/yolo26s/weights/best.pt",
}
MAP50 = {"yolo26n (nano)": 0.790, "yolo26s (small)": 0.813}   # ⚠️ số CŨ — cập nhật từ validation_report.md sau khi train data chicken_counting_v18_yolo mới

assert VIDEO.exists(), f"Thiếu video: {VIDEO}"
for p in MODELS.values():
    assert p.exists(), f"Thiếu weight: {p}"

print(f"ML_ROOT : {ML_ROOT}")
print(f"Device  : {DEVICE}")
print(f"Conf    : {CONF} | IoU: {IOU} | LIVE: {SHOW_LIVE}")

ML_ROOT : /home/loc-nguyen/PROJECT/AI/Automated-ChickenFarm-Management/04_machine_learning
Device  : cuda
Conf    : 0.3 | IoU: 0.5 | LIVE: False


In [20]:
# ══ Load 2 model ══
models, meta = {}, {}
for name, path in MODELS.items():
    m = YOLO(str(path))
    m.to(DEVICE)
    models[name] = m
    meta[name] = {
        "size_mb": round(path.stat().st_size / 1024**2, 1),
        "classes": list(m.names.values()),
        "map50": MAP50[name],
    }
    print(f"{name:16s} | {path}")
    print(f"{'':16s} | size={meta[name]['size_mb']} MB | classes={meta[name]['classes']} | mAP@0.5={MAP50[name]}")

yolo26n (nano)   | /home/loc-nguyen/PROJECT/AI/Automated-ChickenFarm-Management/04_machine_learning/models/chicken_detection/runs/yolo26n/weights/best.pt
                 | size=5.2 MB | classes=['broiler'] | mAP@0.5=0.79
yolo26s (small)  | /home/loc-nguyen/PROJECT/AI/Automated-ChickenFarm-Management/04_machine_learning/models/chicken_detection/runs/yolo26s/weights/best.pt
                 | size=19.4 MB | classes=['broiler'] | mAP@0.5=0.813


In [21]:
# ══ Thông tin video ══
cap = cv2.VideoCapture(str(VIDEO))
assert cap.isOpened(), f"Không mở được {VIDEO}"
vfps   = cap.get(cv2.CAP_PROP_FPS) or 30.0
nfr    = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
w      = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h      = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
cap.release()

print(f"File    : {VIDEO.name}")
print(f"Độ phân giải : {w}×{h}")
print(f"FPS gốc      : {vfps:.2f}")
print(f"Số frame     : {nfr} (~{nfr/vfps:.1f} giây)")

File    : video_test.mp4
Độ phân giải : 1280×720
FPS gốc      : 30.00
Số frame     : 927 (~30.9 giây)


In [22]:
# ══ Hàm lõi: inference + vẽ bbox + HUD (FPS / Threshold / Counting) ══
BOX_COLOR = (80, 220, 80)      # xanh lá
TXT_COLOR = (255, 255, 255)
BAR_COLOR = (20, 20, 20)


def timed_predict(model, frame, conf=CONF, iou=IOU):
    """Chạy inference có đo thời gian (đồng bộ CUDA để đúng)."""
    if DEVICE == "cuda":
        torch.cuda.synchronize()
    t0 = time.perf_counter()
    res = model.predict(frame, conf=conf, iou=iou, verbose=False, device=DEVICE)[0]
    if DEVICE == "cuda":
        torch.cuda.synchronize()
    dt = (time.perf_counter() - t0) * 1000.0          # ms
    boxes = res.boxes
    xyxy = boxes.xyxy.cpu().numpy() if len(boxes) else np.empty((0, 4))
    confs = boxes.conf.cpu().numpy() if len(boxes) else np.empty((0,))
    return xyxy, confs, dt


def draw_half(frame, xyxy, confs, hud, scale=1.0):
    """Vẽ bbox + thanh HUD lên 1 khung hình (copy)."""
    out = frame.copy()
    # bbox
    for (x1, y1, x2, y2), c in zip(xyxy, confs):
        p1 = (int(x1 * scale), int(y1 * scale))
        p2 = (int(x2 * scale), int(y2 * scale))
        cv2.rectangle(out, p1, p2, BOX_COLOR, 2)
        label = f"{c:.2f}"
        cv2.putText(out, label, (p1[0], max(14, p1[1] - 5)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.45, BOX_COLOR, 1, cv2.LINE_AA)
    # HUD: nền bán trong suốt, 5 dòng
    lines = [
        hud["title"],
        f"FPS      : {hud['fps_ema']:5.1f}  (avg {hud['fps_avg']:5.1f})",
        f"Threshold: {hud['conf']:.2f}",
        f"Counting : {hud['count']}  (max {hud['count_max']}, total {hud['total']})",
        f"Latency  : {hud['ms']:5.1f} ms",
    ]
    pad, lh = 6, 20
    bh = lh * len(lines) + pad * 2
    overlay = out.copy()
    cv2.rectangle(overlay, (0, 0), (out.shape[1], bh), BAR_COLOR, -1)
    cv2.addWeighted(overlay, 0.6, out, 0.4, 0, out)
    for i, s in enumerate(lines):
        cv2.putText(out, s, (8, pad + lh * (i + 1) - 5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5,
                    BOX_COLOR if i == 0 else TXT_COLOR, 1, cv2.LINE_AA)
    return out

In [23]:
# ══ CHẠY: mỗi frame → cả 2 model → video chia đôi (trái=nano, phải=small) ══
FPS_SMOOTH = 0.90   # hệ số EMA (càng cao càng mượt)

stats = {n: {"ms": [], "count": [], "total": 0, "fps_ema": 0.0} for n in models}

cap = cv2.VideoCapture(str(VIDEO))
assert cap.isOpened()
writer = cv2.VideoWriter(str(OUTPUT),
                         cv2.VideoWriter_fourcc(*"mp4v"),
                         vfps, (HALF_W * 2, HALF_H))

# warmup GPU 3 lần đầu (không tính vào số liệu)
dummy = np.zeros((HALF_H, HALF_W, 3), np.uint8)
for _ in range(3):
    for m in models.values():
        timed_predict(m, dummy)

frames_done, t_start = 0, time.perf_counter()
try:
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        halves = []
        for name, m in models.items():
            st = stats[name]
            xyxy, confs, ms = timed_predict(m, frame)
            fps_inst = 1000.0 / ms if ms > 0 else 0.0
            st["fps_ema"] = (fps_inst if frames_done == 0
                             else FPS_SMOOTH * st["fps_ema"] + (1 - FPS_SMOOTH) * fps_inst)
            st["ms"].append(ms)
            st["count"].append(len(xyxy))
            st["total"] += len(xyxy)

            half = cv2.resize(frame, (HALF_W, HALF_H))
            scale = HALF_W / frame.shape[1]
            hud = {
                "title": f"{name}  [{meta[name]['size_mb']} MB]",
                "fps_ema": st["fps_ema"],
                "fps_avg": (1000.0 * (frames_done + 1) / sum(st["ms"])),
                "conf": CONF,
                "count": len(xyxy),
                "count_max": max(st["count"]),
                "total": st["total"],
                "ms": ms,
            }
            halves.append(draw_half(half, xyxy, confs, hud, scale))

        canvas = np.hstack(halves)
        # nhãn 2 nửa
        cv2.putText(canvas, "YOLO26n", (10, canvas.shape[0] - 12),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 0), 2, cv2.LINE_AA)
        cv2.putText(canvas, "YOLO26s", (HALF_W + 10, canvas.shape[0] - 12),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 0), 2, cv2.LINE_AA)
        writer.write(canvas)
        frames_done += 1

        if SHOW_LIVE:
            cv2.imshow("compare  [q = thoat]", canvas)
            if cv2.waitKey(1) & 0xFF == ord("q"):
                break
        if frames_done % 150 == 0:
            el = time.perf_counter() - t_start
            print(f"  ... {frames_done}/{nfr} frame | {el:.1f}s | "
                  f"n={stats['yolo26n (nano)']['fps_ema']:.1f} FPS, "
                  f"s={stats['yolo26s (small)']['fps_ema']:.1f} FPS", flush=True)
finally:
    cap.release()
    writer.release()
    if SHOW_LIVE:
        cv2.destroyAllWindows()

wall = time.perf_counter() - t_start
print(f"\n✓ Xong {frames_done} frame trong {wall:.1f}s → output: {OUTPUT.name}")

  ... 150/927 frame | 2.9s | n=182.7 FPS, s=147.5 FPS
  ... 300/927 frame | 5.8s | n=176.0 FPS, s=143.4 FPS
  ... 450/927 frame | 8.7s | n=186.9 FPS, s=148.4 FPS
  ... 600/927 frame | 11.6s | n=177.6 FPS, s=143.7 FPS
  ... 750/927 frame | 14.5s | n=176.7 FPS, s=145.1 FPS
  ... 900/927 frame | 17.4s | n=181.4 FPS, s=145.2 FPS

✓ Xong 927 frame trong 17.9s → output: output_compare.mp4


In [24]:
# ══ BẢNG SO SÁNH — in ra terminal + lưu CSV ══
rows = []
for name, st in stats.items():
    ms = np.array(st["ms"], float)
    cnt = np.array(st["count"], int)
    fps = 1000.0 / ms
    rows.append({
        "model": name,
        "size_mb": meta[name]["size_mb"],
        "mAP@0.5": meta[name]["map50"],
        "avg_ms": round(ms.mean(), 2),
        "ms_std": round(ms.std(), 2),
        "fps_avg": round(fps.mean(), 1),
        "fps_min": round(fps.min(), 1),
        "fps_max": round(fps.max(), 1),
        "count_avg": round(cnt.mean(), 1),
        "count_max": int(cnt.max()),
        "total_dets": st["total"],
    })

df = pd.DataFrame(rows)
df.to_csv(CSV_OUT, index=False, encoding="utf-8-sig")

print("=" * 88)
print("  SO SÁNH 2 MODEL TRÊN VIDEO THẬT  (conf =", CONF, "| IoU =", IOU, ")")
print("=" * 88)
print(df.to_string(index=False))
print("-" * 88)

# FPS tổng của toàn bộ pipeline song song (1 frame = chạy cả 2 model)
per_frame = np.sum([np.array(stats[n]["ms"]) for n in stats], axis=0)
print(f"Pipeline song song : {(1000.0 / per_frame).mean():.1f} FPS "
      f"({per_frame.mean():.1f} ms/frame cho CẢ 2 model)")
print(f"Video gốc          : {vfps:.0f} fps · Kết quả lưu: {OUTPUT.name} · CSV: {CSV_OUT.name}")
df

  SO SÁNH 2 MODEL TRÊN VIDEO THẬT  (conf = 0.3 | IoU = 0.5 )
          model  size_mb  mAP@0.5  avg_ms  ms_std  fps_avg  fps_min  fps_max  count_avg  count_max  total_dets
 yolo26n (nano)      5.2    0.790    5.60    0.95    181.3     64.0    205.1        9.5         17        8801
yolo26s (small)     19.4    0.813    6.93    0.53    144.9     70.0    155.7       23.1         34       21398
----------------------------------------------------------------------------------------
Pipeline song song : 80.3 FPS (12.5 ms/frame cho CẢ 2 model)
Video gốc          : 30 fps · Kết quả lưu: output_compare.mp4 · CSV: comparison.csv


,model,size_mb,mAP@0.5,avg_ms,ms_std,fps_avg,fps_min,fps_max,count_avg,count_max,total_dets
0,yolo26n (nano),5.2,0.790,5.60,0.95,181.3,64.0,205.1,9.5,17,8801
1,yolo26s (small),19.4,0.813,6.93,0.53,144.9,70.0,155.7,23.1,34,21398


## Cách đọc kết quả

- **FPS**: `fps_avg` = hiệu năng trung bình khi chạy video thật (khác với benchmark ảnh rời trong `validation_report.md`). Nano kỳ vọng nhanh hơn small; nếu cả 2 < 30 fps nghĩa là chưa real-time trên máy này.
- **Threshold**: đổi `CONF` ở ô Config rồi chạy lại → số `count` thay đổi (conf cao → ít bbox nhưng chắc chắn hơn).
- **Counting**: `count_avg` = số gà trung bình mỗi khung hình; `total_dets` = tổng bbox cả video (frame nào cũng đếm lại → không phải số gà duy nhất).
- **So sánh chất lượng**: đối chiếu `mAP@0.5` (đang là số của model cũ — cập nhật `MAP50` sau khi train data mới) — small chính xác hơn nhưng chậm hơn.

Video đầu ra: `testing/output_compare.mp4` — nửa trái **yolo26n**, nửa phải **yolo26s**, mỗi nửa có HUD FPS / Threshold / Counting.